# 02 — Unstructured → structured data

Turn messy files into queryable Spark DataFrames.

**Before running**, generate the practice files from the project root:

```powershell
python scripts/generate_unstructured_data.py
```

That writes three files into `data/generated/`:

| File | Messiness |
|------|-----------|
| `events.jsonl` | Nested JSON lines; some `props` are strings instead of objects; some users missing |
| `app.log` | Free-text logs in **two** slightly different formats |
| `support_tickets.csv` | CSV with pipe-delimited tags + free-text bodies that hide IDs |

Work top to bottom. Each exercise builds on the previous. Suggested approach for every task:

1. Load the raw data and look at a few rows.
2. Parse / extract fields into typed columns.
3. Answer the questions with normal DataFrame ops (`filter`, `groupBy`, etc.).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("unstructured-to-structured")
    .master("local[*]")
    .getOrCreate()
)

DATA_DIR = "../data/generated"
print(f"Spark version: {spark.version}")


## Exercise A — Nested JSON event logs (`events.jsonl`)

Each line is one JSON event. Nesting means fields like `user.id` and `device.type` aren't flat columns yet.

### Goals
1. Read the file as JSON (hint: `spark.read.json(...)` works on `.jsonl`).
2. Flatten into columns: `event_id`, `ts`, `user_id`, `session`, `page`, `device_type`, `browser`, `referrer`, `cart_value`.
3. Parse `ts` into a real timestamp.
4. ~5% of rows store `props` as a **string** (JSON-inside-a-string). Detect those and parse them so `referrer` / `cart_value` aren't null just because of that mess.
5. Answer:
   - How many events have a missing `user_id`?
   - Top 5 pages by event count
   - Average `cart_value` by `device_type` (ignore null cart values)


In [ ]:
# A0 — inspect the raw JSON (schema will show nested structs / odd types)
raw_events = spark.read.json(f"{DATA_DIR}/events.jsonl")
raw_events.printSchema()
raw_events.show(3, truncate=False)


In [ ]:
# A1 — YOUR TURN: flatten + clean into a queryable `events` DataFrame
# Hints:
#   F.col("user.id"), F.from_json(...), F.to_timestamp(...)
#   For stringified props, check the schema of `props` — it may be a mix.
#   One approach: if props is a string column, from_json it; if it's a struct, use it directly.
#   With this generator, Spark often infers `props` as STRING because of the dirty rows —
#   so from_json on the whole column is a good first try.

props_schema = T.StructType([
    T.StructField("referrer", T.StringType()),
    T.StructField("cart_value", T.DoubleType()),
])

# events = ...
# events.printSchema()
# events.show(5, truncate=False)


In [ ]:
# A2 — YOUR TURN: answer the three questions
# missing_users = ...
# top_pages = ...
# avg_cart = ...


## Exercise B — Free-text logs with regex (`app.log`)

These aren't CSV or JSON — just lines of text in two formats:

```
2024-07-03 14:22:11 [INFO] service=api request_id=req-123456 handled request status=200 latency_ms=42
2024-07-03T14:22:11Z level=ERROR svc=payments rid=req-654321 request failed status=500 latency_ms=1201 err=timeout
```

### Goals
1. Read as text: `spark.read.text(...)` → one column named `value`.
2. Use `F.regexp_extract` (and maybe a couple of patterns) to pull out:
   `ts`, `level`, `service`, `request_id`, `status`, `latency_ms`.
3. Cast `status` / `latency_ms` to integers; parse `ts` to timestamp (both formats).
4. Answer:
   - Error rate: `% of lines where level = ERROR`
   - p95-ish latency: what's the 95th percentile of `latency_ms`? (`F.percentile_approx`)
   - Which `service` has the most ERROR lines?


In [ ]:
# B0 — peek at raw lines
raw_logs = spark.read.text(f"{DATA_DIR}/app.log")
raw_logs.show(5, truncate=False)


In [ ]:
# B1 — YOUR TURN: parse into a structured `logs` DataFrame
# Hints:
#   F.regexp_extract(F.col("value"), r"pattern", 1)
#   Level appears either as [INFO] or level=INFO
#   Service appears as service=... or svc=...
#   request_id as request_id=... or rid=...
#   For timestamps, try coalescing two to_timestamp formats:
#     "yyyy-MM-dd HH:mm:ss" and "yyyy-MM-dd'T'HH:mm:ss'Z'"

# logs = ...
# logs.show(5, truncate=False)


In [ ]:
# B2 — YOUR TURN: error rate, p95 latency, noisiest service
# ...


## Exercise C — Free-text + multi-value fields (`support_tickets.csv`)

Looks like a normal CSV, but two columns are semi-structured:

- `tags`: pipe-delimited (`billing|urgent|refund`) — sometimes blank
- `body`: prose with buried `customer_id=...` and sometimes `order_id=...`

### Goals
1. Load with `header=True, inferSchema=True`.
2. Explode tags into one row per tag (`F.split` + `F.explode_outer` so blank tags still keep the ticket).
3. Extract `customer_id` and `order_id` from `body` with regex; cast to int (null if missing).
4. Answer:
   - Top 5 tags by ticket count
   - How many tickets mention an `order_id`?
   - Join extracted `customer_id` to `customers.csv` — how many tickets reference a real customer? How many don't?


In [ ]:
# C0 — inspect
tickets = spark.read.csv(f"{DATA_DIR}/support_tickets.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)
tickets.show(3, truncate=80)


In [ ]:
# C1 — YOUR TURN: structured tickets + exploded tags
# Hints:
#   F.split("tags", r"\|")
#   F.explode_outer(...)
#   F.regexp_extract(F.col("body"), r"customer_id=(\d+)", 1)
#   empty string from regexp_extract → cast carefully ("" won't become null by itself;
#   use F.when(col == "", None) before cast, or nullif)

# tickets_clean = ...
# tickets_by_tag = ...


In [ ]:
# C2 — YOUR TURN: tag counts, order_id coverage, customer match rate
# ...


## Stretch goals (optional)

Pick one if you want a harder pass:

1. **Unify formats with a schema.** Define an explicit schema for events (`StructType`) and re-read with `spark.read.schema(...).json(...)`. Compare inferred vs explicit — what broke?
2. **Write curated Parquet.** Save your cleaned `events`, `logs`, and `tickets_clean` under `../output/curated/` as Parquet. Re-read them and confirm queries no longer need regex.
3. **Build a mini incident report.** From `logs`, find ERROR bursts: for each service, count ERRORs per hour (`F.date_trunc("hour", ts)`). Which hour was worst?
4. **End-to-end join.** Tickets → extracted `order_id` → `orders.csv` → `products.csv`. What's the most common product category among tickets that mention an order?

When you're done:

```python
spark.stop()
```


In [ ]:
# spark.stop()
